# 01 우동호 외(2023) 마법공식 재현

## 0. 원문 정리 (원문을 먼저 읽고 옮김)
원문: 우동호·최흥식·김선웅(2023), 「한국주식시장에서의 마법공식 투자전략 성과분석」, 한국산학기술학회논문지 24(3), 508–519. 3.1·3.2절(p.512–513), 표 1~7(p.512–516).

| 항목 | 원문 | 쪽 | 우리 설정 (`공통/포트폴리오.py`의 `우동호2023_원문`) |
|---|---|---|---|
| 대상 시장 | 유가증권시장 + 코스닥시장 | 512 | `시장=["KS","KQ"]` (CP10000300) |
| 제외 업종 | 은행·보험·증권·투자금융·신용금고·종합금융·기타금융 | 512 | `("금융업", {"이름": "원문"})`: 그날 거래소 업종명(CP10001090)에 은행·보험·증권·금융·금고가 있으면 뺌. 거래소 업종이 빈 날은 이름(지금 이름)에 은행·보험·증권·금융·금고·캐피탈·카드·저축·신금이 있거나 그날 WICS 대분류(CP10000600)가 '금융'이면 뺌(`종목거르기.금융업(이름="원문")`). 업종이 빈 날의 판정은 원문에 없어 원문 7업종을 보조 자료(이름·WICS)로 대응(기본값, 11절). 투자기구(리츠·선박·부동산투자) 제외는 원문에 없어 뺌(회원 05:57) |
| 결산월 | 12월 결산법인만 | 512 | `("결산월", {"월": 12})` (회계연도별 결산월 이력 CP10002200) |
| 거래정지 | 거래일 기준 시점에 거래중지면 제외 | 512 | `거래정지` (일간 S410002600) |
| 가격 | 액면분할 등이 반영된 수정주가 | 512 | `가격=수정주가` (S410000700, 배당 미포함) |
| 리밸런싱 | 매년 4월 첫 거래일, 1년 보유 후 전량 교체 | 512–513 | `기준일="첫거래일"`, `리밸런싱=12`, `전량교체=True` |
| 재무 자료 | 직전 연도(t−1) 결산자료('직전년도에 공시된 재무제표', '2003년에서 2020년 결산자료') | 512 | `재무시점=("결산후개월", 3)`, `재무연도="직전년도"`: 결산일 + 3개월(3월 말)이 지난 **회계연도 t−1 결산만**. t−1 결산이 없는 종목은 그해 빠짐(2026-11-06 원문대로 고침: 전에는 2년 안 가장 최근 행이라 FY t−2도 씀) |
| ROC | EBIT / (순운전자본 + 순고정자산), 순운전자본 = 유동자산 − 유동부채, 순고정자산 = 총자산 − 무형자산 | 512 | `지표.ROC` (무형자산 M000901024) |
| EBIT | 법인세차감전이익 + 이자비용 | 512 | `지표.EBIT` = 세전계속사업이익 M000907001 + 이자비용 M000906020 |
| EY | EBIT / 기업가치, 기업가치 = 시가총액 + 부채총액 − 현금 | 512 | `지표.EY` (시가총액 = 4월 첫 거래일 값) |
| 순위 | ROC·EY 각각 내림차순 순위 → 합산 | 512 | `합산="순위합산"` |
| 종목 수·비중 | 총 순위 상위 30종목 동일비중 | 512 | `고르기=("상위N", 30)`, `비중=["동일"]` |
| IMF(개선 공식) | ROC 대신 GP/A | 513 | `지표=[("GPA","높게"),("EY","높게")]`, GP/A = 매출총이익(M000904007) / 자산총계 (원문 p.513 'GP/A는 매출총이익을 총자산으로 나누어 계산한다', Novy-Marx 2013) |
| 거래비용 | 수수료 0.015%, 거래세 0.25%, 매수·매도마다 슬리피지 0.5% | 512 | `거래비용=dict(수수료=0.00015, 거래세=0.0025, 슬리피지=0.005)` |
| 무위험이자율 | 3년 만기 국고채 | 513 | 월(회귀): `무위험="E11.02.003.008"`(국고3년 월말, 연 %/1200, 직전 달 값). 일(샤프): `읽기.국고3년일간()`(X02 국고3년 일간, 연 %/100/365, 직전 날 값. /365는 원문 KOSPI 샤프 1.34가 나오는 쪽, `_검증/01_정의역산_결과.txt`) |
| 벤치마크 | KOSPI | 513 | `시장지수="I.001"` |
| EBIT 서술(2.3절) | 영업이익에서 비영업활동 손익을 빼고 영업외·특별손익 중 영업활동 손익은 더한 금액, 법인세차감전이익에 이자비용을 더함 | 511 | 3.2절 식과 같은 끝(세전계속사업이익 + 이자비용) |
| 표본 표(표 1) | 연도별 Number·Average return·Standard deviation(%) | 512 | Number·평균수익률·표준편차 모두 8절에서 대조(평균 내는 법은 원문에 없어 기본값, 11절), 결과 `01_표1대조.csv` |
| 원문 판정 | 본문 글의 방향·유의(별표) 문장(p.514–517) | 514–517 | 7절 `01_원문판정.csv`: 원문 문장마다 원문 방향과 우리 방향을 나란히(유의 = OLS t를 원문 별표 수준 10%·5%·1%와 비교) |
| 성과 지표 | 누적수익률, 표준편차, MDD, 샤프, 정보비율, 젠센 알파, FF3, Carhart 4요인 | 513 | 누적·알파(t는 OLS) = 월수익률 `평가.성과표`. SD·Sharpe·IR·MDD = 일간 수익률 `평가.일간정의`(일 표준편차 × √N 등. 원문에 정의가 없고, 원문 KOSPI 숫자 83.49%·1.34·−54.54%가 이 정의에서 나옴, 9절) |
| 요인 | SMB = 시총 하위30% − 상위30%, HML = BM 상위30% − 하위30%, MOM = 과거 11개월 상위30% − 하위30% | 513 | 요인은 과제 공지대로 DataGuide 요인 사용: `평가.요인`의 SMB = 3FM.2B3.SMB, HML = 3FM.2B3.HML, MOM = 3FM.2M3.MOM(DataGuide 2×3 요인지수, 기준일 간 종가 비율). 원문 p.513은 30/30 직접 구성, 그래서 원문 3·4요인 α와 정의가 다름 |
| 기간 | 2004-04 ~ 2022-03 (2003~2020 결산자료) | 512 | `기간=("2004-04","2022-03")`, `"끝": "마지막거래일"` (마지막 구간 = 2022-03 첫 거래일 → 2022-03-31, 원문대로. 2026-11-04 고침, 전에는 2022-04-01까지라 하루 더 있었음) |
| 규모별 | 전체 종목(= 표본 전체, p.513 '4.1 전 종목 대상 …' 아래 'Fig. 1은 표본 전체를 대상으로')을 시가총액 5분위로 나눠 분위마다 따로. 표본 = '필요한 재무자료 및 개별증권자료를 입수 가능한 기업'(p.512) | 510·512–516 | `("지표있음", {"지표들": MF는 ROC·EY, IMF는 GPA·EY})` 뒤 `("시총분위", {"분위": q})` (2026-11-05 2차 진단으로 고침: 전에는 자료 없는 종목까지 넣고 나눴음. '필요한 자료'가 어느 항목인지는 원문에 없어 전략별로 읽음, 다른 두 읽기는 `_검증/2차_01b_분위모집단_결과.txt`) |

원문 안에서 서로 맞지 않는 곳(우리가 고치지 않음, 0단계 대조 2026-11-04): p.512 본문은 2020년 평균수익률 96.69%, 표 1은 98.69 / p.517 결론의 '4분위 마법공식 (−) 알파'는 표 6(4분위)에는 음수가 없고 표 7(5분위) MF 4F −0.0003뿐 / p.514 '전 종목을 월별로 집계한 분석에서 … KOSPI 지수는 212.39% 증가하였고, MDD는 –54.54%, 표준편차는 83.49%, 샤프비율은 1.34'라고 쓰지만 이 KOSPI 숫자는 월수익률로는 나오지 않고 일간 수익률(× √일 수)에서만 나온다(`_검증/01_정의역산_결과.txt`, 9절).

줄임말: EW(동일가중, Equal Weight), VW(시가총액가중, Value Weight), LS(롱숏: 좋은 분위를 사고 나쁜 분위를 판 수익률),
MDD(최대 낙폭, Maximum Drawdown), IR(정보비율, Information Ratio), CAPM(자본자산가격결정모형), FF3(파마-프렌치 3요인), Carhart(카하트 4요인: FF3 + 모멘텀),
OLS(보통최소제곱).

**원문에 없는 추가는 이 노트북에서 뺐다**(2026-11-06 회원 결정 "원문에 말이 없으면 빼. 원문에 있으면 원문대로 해."): VW, 확인판(발표일 + 150일), 5분위·LS, 월간 × √12 성과 칸, 월 누적 대 일간 누적 표, 단계별 표·이상값·기록 표, 원문 차이 진단 → `연구/따라하기에서_옮김_01.ipynb`(맨 앞에 옮긴 목록).

## 1. 불러오기
계산은 모두 `공통/` 폴더의 모듈이 한다. 이 노트북은 설정을 정하고, 실행하고, 결과를 본다.

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "공통"))
import numpy as np
import pandas as pd
import 데이터읽기 as 읽기, 지표, 포트폴리오 as 포, 평가
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
시험 = os.environ.get("FOLLOW_TEST") == "1"  # 짧은 시험: 기간 2년, 결과는 _검증/1106/시험결과/에만(실제 결과 파일을 덮어쓰지 않음)
if 시험:
    평가.결과폴더 = Path.cwd().parent / "_검증" / "1106" / "시험결과"
    평가.결과폴더.mkdir(parents=True, exist_ok=True)
print("시험 모드:", 시험, "/ 결과 저장 폴더:", 평가.결과폴더)

## 2. 데이터 경로와 기간
데이터를 어디서 읽을지, 어느 기간을 돌릴지 정한다. 기간은 (첫 리밸런싱 월, 마지막 기준일 월)이다.

In [ ]:
# 새로 통째로 받은 파일(읽기.DATA 기본 = data\통합_전체_202611, DataGuide 달력기준 일간)로 원문 기간을 돌린다.
기간 = ("2004-04", "2022-03")  # 원문 p.512 2004년 4월~2022년 3월. 마지막 구간은 2022-03 마지막 거래일에서 끊음(아래 "끝")
if 시험:
    기간 = ("2004-04", "2006-03")  # 짧은 시험: 첫 달부터 2년
print("데이터:", 읽기.DATA, "/ 기간:", 기간)

## 3. 설정
원문대로 '4월 첫 거래일, 직전 연도 결산자료'. 12월 결산이면 결산일 + 3개월(3월 말)이 지나면 쓰고(발표일은 보지 않음), 회계연도 t−1 결산만 쓴다(`재무연도="직전년도"`, 원문 p.512 '직전년도에 공시된 재무제표'). 설정 = `포.우동호2023_원문`(금융업 = 원문 7업종 판정, 투자기구 거르기 없음). 아래 칸이 설정에 이 값이 들어 있는지 확인한다(보통주만·수정주가 포함).

In [ ]:
IMF = {"이름": "우동호2023_IMF", "지표": [("GPA", "높게"), ("EY", "높게")]}
설정 = {"MF": {**포.우동호2023_원문, "기간": 기간, "끝": "마지막거래일"}, "IMF": {**포.우동호2023_원문, **IMF, "기간": 기간, "끝": "마지막거래일"}}
for k, v in 설정.items():  # 원문 p.512 '직전년도', '수정주가', 금융 7업종. 보통주만(회원 결정 2026-11-06). 투자기구 거르기 없음(원문에 없음, 회원 05:57)
    assert v["재무연도"] == "직전년도" and v["가격"] == 읽기.수정주가 and "보통주" in v["제외"] and v["비중"] == ["동일"], k
    assert ("금융업", {"이름": "원문"}) in v["제외"] and "투자기구" not in v["제외"], k
pd.DataFrame(설정).astype(str)

## 4. 본문: 상위 30종목 EW (원문 표 2)
4월 첫 거래일마다 거르기 → 지표 계산 → 순위 합산 → 상위 30종목을 동일비중으로 사서 1년 보유한다(원문 p.512).
- **요인**: 과제 공지대로 DataGuide 요인을 쓴다(`평가.요인`: SMB = 3FM.2B3.SMB, HML = 3FM.2B3.HML, MOM = 3FM.2M3.MOM). 원문 p.513은 SMB = 시가총액 하위 30% − 상위 30%, HML = 장부가치/시장가치 상위 30% − 하위 30%, MOM = 과거 11개월 수익률 상위 30% − 하위 30%로 직접 구성해, 원문 3·4요인 α와 정의가 다르다. 시장 요인 = KOSPI − 무위험(국고 3년).
- **누적·알파**는 월수익률(원문 p.514 '월별로 집계'), t는 OLS.
- **SD·Sharpe·IR·MDD**는 같은 보유 비중을 일간 수익률에 넣어 잰다(`평가.일간정의`: 일 표준편차 × √N, 샤프 = 일 초과수익 평균/표준편차 × √N, IR = KOSPI 대비 같은 꼴, MDD = 일간 가치 최대 낙폭. 원문에 정의가 없고 KOSPI 원문 숫자가 이 정의에서 나옴, 9절 KOSPI 줄).
- KOSPI 행의 알파·t는 시장을 자기 자신에 회귀한 값이라 뜻이 없어 빈칸으로 둔다(원문 표도 빈칸).

In [ ]:
결과 = {k: 포.실행(v) for k, v in 설정.items()}
g = 포.기준일(설정["MF"])
F = 평가.요인(g, 설정["MF"])  # 시장 = KOSPI(I.001), RF = 국고 3년(E11.02.003.008) 연 %/1200 직전 달 값, MKT = 시장 − RF. SMB·HML·MOM = DataGuide 요인(과제 공지)
R = pd.DataFrame({k: o["수익률"][("동일", "상위30")] for k, o in 결과.items()})
R["KOSPI"] = F["시장"]
R = R.dropna(how="all")

# 일간: 같은 보유 비중을 일간 수익률(월간 실행과 같은 '쓰지 말 것' 규칙)에 넣는다
가격 = 설정["MF"]["가격"]
날 = 읽기.상장가격(가격).loc[g[0]:g[-1]].index
Rd = 포.쓰지말것칸(포.기간수익률(읽기.상장가격(가격), 날), 가격)
국고 = 읽기.국고3년일간()  # 원문 p.513 '3년 만기 국고채 수익률'(연 %)
무위험일 = 국고.reindex(날.union(국고.index)).ffill().reindex(날).shift(1) / 100 / 365  # 직전 날 값
시장일 = 읽기.지수()[설정["MF"]["시장지수"]].reindex(날).pct_change(fill_method=None).iloc[1:]
일 = {k: 포.보유(Rd, o["보유"][("동일", "상위30")], 설정[k]["거래비용"], 설정[k]["전량교체"]) for k, o in 결과.items()} | {"KOSPI": 시장일}

열 = ["Return", "MDD", "SD", "Sharpe", "IR", "Jensen α", "Jensen t", "3F α", "3F t", "4F α", "4F t"]


def 원문꼴(R월, R일):
    """원문 표 2~7 칸: 누적·알파·t(OLS)는 월수익률, SD·Sharpe·IR·MDD는 일간."""
    월 = 평가.성과표(R월, F)
    일표 = pd.DataFrame({k: 평가.일간정의(R일[k], 시장일, 무위험일) for k in R월}).T
    return pd.DataFrame({"Return": 월["누적"], "MDD": 일표["MDD"], "SD": 일표["SD"], "Sharpe": 일표["Sharpe"], "IR": 일표["IR"],
                         "Jensen α": 월["CAPM α(월)"], "Jensen t": 월["CAPM t(OLS)"], "3F α": 월["FF3 α(월)"], "3F t": 월["FF3 t(OLS)"],
                         "4F α": 월["Carhart α(월)"], "4F t": 월["Carhart t(OLS)"]}).astype(float)


표 = 원문꼴(R, 일)
표.loc["KOSPI", ["Jensen α", "Jensen t", "3F α", "3F t", "4F α", "4F t"]] = float("nan")  # 시장을 자기 자신에 회귀한 값(수치 잡음)이라 빈칸(원문 표도 빈칸)
print("월 수", int(R["MF"].notna().sum()), "/ 일 수", len(일["MF"].dropna()))
표

## 5. 누적수익률 그림 (원문 Fig. 1)
원문 Fig. 1처럼 보통(선형) 축, 선은 MF·IMF·KOSPI 셋(p.513). 원문 세 선이 2004-04에 약 100%에서 시작해 세로값 = 누적 가치(1 + 누적수익률) × 100%로 그린다(`_검증/1106/원문사양_01.md` A3). 첫 점은 2004-04 첫 거래일 100%. 선 모양은 원문 범례대로 IMF 실선, MF 파선, KOSPI 점선. 로그 축 그림은 확장연구(`연구/따라하기에서_옮김_01.ipynb`).

In [ ]:
그림R = pd.concat([pd.DataFrame(0.0, index=[g[0]], columns=["IMF", "MF", "KOSPI"]), R[["IMF", "MF", "KOSPI"]]])  # 첫 점 = 2004-04 첫 거래일 100%
평가.누적그림(그림R, "우동호 외(2023) Fig. 1 재현: 상위 30종목 EW", 퍼센트=True, 시작100=True);  # 열 순서 = 선 모양: IMF 실선, MF 파선, KOSPI 점선(원문 범례 '——IMF', '-----MF', '······KOSPI')

## 6. 시가총액 5분위별 (원문 표 3~7)
4월 첫 거래일에 본문 거르기를 한 뒤, 그 전략의 지표 자료가 있는 종목(MF = ROC·EY, IMF = GPA·EY; 원문 '표본', p.512·513)만 4월 첫 거래일 시가총액으로 5분위로 나누고(1분위 = 가장 작음, 같은 수), 분위마다 따로 상위 30종목을 고른다(원문 p.514 '시가총액 하위 20%' 등). 분위를 나누는 모집단은 원문에 없어 기본값(11절). 다른 모집단 판은 확장연구.
월 누적과 일간 누적이 같은지 보는 표(원문에 없음)는 확장연구(`연구/따라하기에서_옮김_01.ipynb` 15절)로 옮겼다.

In [ ]:
# 분위 모집단 = 그 전략 지표 자료가 있는 종목(원문 '표본', p.512·513). 2026-11-05 2차 진단 1
규모결과 = {f"{k}_시총{q}분위": 포.실행({**설정[k], "제외": 설정[k]["제외"] + [("지표있음", {"지표들": [m[0] for m in 설정[k]["지표"]]}), ("시총분위", {"분위": q})]}) for q in range(1, 6) for k in ["MF", "IMF"]}
규모 = pd.DataFrame({n: o["수익률"][("동일", "상위30")] for n, o in 규모결과.items()})
규모일 = {n: 포.보유(Rd, o["보유"][("동일", "상위30")], o["설정"]["거래비용"], o["설정"]["전량교체"]) for n, o in 규모결과.items()}
규모표 = 원문꼴(규모, 규모일)
규모표

## 7. 원문 vs 우리 (원문 표 2~7)
원문 숫자는 표 2~7(p.514–516)을 옮겨 적은 것이고, 우리 숫자는 위 4·6절 표에서 코드가 가져온다. 누적·MDD·표준편차는 소수(980.35% → 9.8035). 두 칸의 단위는 같다(SD·Sharpe·IR·MDD 모두 일간 정의, 4절).

**원문 판정**(두 번째 표, `01_원문판정.csv`): 원문 본문 글(p.514–517)이 말한 방향·크기만 대조한다. '유의'는 원문 별표 수준을 그대로 OLS t에 쓴다: 1% = |t| ≥ 2.58, 5% = 1.96, 10% = 1.645(원문 표 5 MF 젠센 0.0049*(1.68)을 p.515가 '10%수준에서 유의'라고 씀). '유의하지 않다' = |t| < 1.645. 원문이 말하지 않은 p 기준 판정(경계 등)은 하지 않는다. KOSPI 대비 문장(p.515–516 4분위 '좋은 성과를 나타내지 못하거나 오히려 나쁜 성과', 개선된 마법공식 '여전히 기존 마법공식 및 벤치마크 보다 좋은 성과', p.516–517 1·2분위 '누적수익률, 샤프비율, 정보비율 등 모든 면에서 월등')은 방향만 본다(비교한 숫자는 '원문 값'·'우리 값' 칸, KOSPI = 같은 기간 표 2 줄).

In [ ]:
원문표 = pd.DataFrame({  # 우동호 외(2023) 표 2~7
    ("전체", "MF"): [9.8035, -0.5034, 0.9479, 2.42, 1.48, 0.0068, 2.32, -0.0050, -1.28, -0.0053, -1.34],
    ("전체", "IMF"): [10.2791, -0.4962, 0.8857, 2.58, 1.57, 0.0071, 2.47, -0.0028, -0.73, -0.0034, -0.87],
    ("전체", "KOSPI"): [2.1239, -0.5454, 0.8349, 1.34, 0.00] + [None] * 6,
    ("시총1분위", "MF"): [44.2427, -0.5760, 1.1082, 3.52, 2.72, 0.0137, 3.88, 0.0139, 3.51, 0.0141, 3.55],
    ("시총1분위", "IMF"): [101.3052, -0.6203, 1.1828, 4.06, 3.31, 0.0179, 4.39, 0.0167, 3.62, 0.0168, 3.64],
    ("시총2분위", "MF"): [17.6391, -0.5155, 1.0596, 2.78, 1.94, 0.0095, 2.76, 0.0097, 2.70, 0.0096, 2.69],
    ("시총2분위", "IMF"): [28.4832, -0.5343, 1.1032, 3.13, 2.32, 0.0120, 3.13, 0.0127, 3.25, 0.0126, 3.24],
    ("시총3분위", "MF"): [6.3007, -0.4963, 0.9663, 1.99, 1.07, 0.0049, 1.68, 0.0044, 1.46, 0.0044, 1.45],
    ("시총3분위", "IMF"): [9.8866, -0.4799, 0.9275, 2.47, 1.51, 0.0068, 2.38, 0.0066, 2.22, 0.0066, 2.21],
    ("시총4분위", "MF"): [2.0623, -0.5528, 0.9325, 1.10, 0.14, 0.0009, 0.33, 0.0005, 0.18, 0.0004, 0.13],
    ("시총4분위", "IMF"): [2.7631, -0.5583, 0.9242, 1.32, 0.35, 0.0020, 0.68, 0.0031, 0.99, 0.0028, 0.90],
    ("시총5분위", "MF"): [3.3954, -0.4398, 0.7890, 1.59, 0.47, 0.0022, 1.13, 0.0004, 0.20, -0.0003, -0.15],
    ("시총5분위", "IMF"): [3.7548, -0.3496, 0.6953, 1.82, 0.54, 0.0028, 1.47, 0.0018, 0.86, 0.0011, 0.57],
}, index=열).T
우리 = {("전체", k): 표.loc[k] for k in ["MF", "IMF", "KOSPI"]}
우리 |= {(f"시총{q}분위", k): 규모표.loc[f"{k}_시총{q}분위"] for q in range(1, 6) for k in ["MF", "IMF"]}
우리표 = pd.DataFrame(우리).T[열].astype(float)
print("우리 = 본문 설정 / 기간", 기간, "/ 월 수", int(R["MF"].notna().sum()), "/ 일 수", len(일["MF"].dropna()))
대조 = pd.concat({"원문": 원문표, "우리": 우리표}, axis=1).swaplevel(axis=1)[열]
대조.to_csv(평가.결과폴더 / "01_원문대조.csv", encoding="utf-8-sig")  # 결과요약 노트북이 읽음
display(대조)

# 원문 판정: (구간, 전략, 대조 칸, 원문 방향, 원문 유의, 원문 쪽·글)
수준 = {"1%": 2.58, "5%": 1.96, "10%": 1.645}
주장 = [("전체", k, a, "+", "5%", "p.514 '젠센의 알파의 계수 값은 … 5% 수준에서 유의한 (+)'") for k in ["MF", "IMF"] for a in ["Jensen"]]
주장 += [("전체", k, a, "−", "아님", "p.514 '3요인 모형과 4요인 모형에서 알파의 계수는 (-)의 값 … 통계적으로 유의하지 않았다'") for k in ["MF", "IMF"] for a in ["3F", "4F"]]
주장 += [(f"시총{q}분위", k, a, "+", "1%", f"p.{514 if q == 1 else 515} {q}분위 '1%수준에서 통계적으로 유의한 (+)'") for q in [1, 2] for k in ["MF", "IMF"] for a in ["Jensen", "3F", "4F"]]
주장 += [("시총3분위", "MF", "Jensen", "+", "10%", "p.515 '0.0049로서 10%수준에서 통계적으로 유의한 (+)'")]
주장 += [("시총3분위", "MF", a, "+", "아님", "p.515 '모두 0.0044로 (+)의 값을 나타냈지만 통계적으로 유의하진 않았다'") for a in ["3F", "4F"]]
주장 += [("시총3분위", "IMF", a, "+", "5%", "p.515 '5%수준에서 통계적으로 유의한 (+)', '5% 수준에서 통계적으로 유의하였다'") for a in ["Jensen", "3F", "4F"]]
주장 += [("시총4분위", k, a, "+", "아님", "p.515 '(+)의 값을 나타냈지만 통계적으로 유의하진 않았다'") for k in ["MF", "IMF"] for a in ["Jensen", "3F", "4F"]]
주장 += [("시총5분위", "MF", a, s, "아님", "p.516 '각각 0.0022, 0.0004로 (+) … 4요인 … –0.0003로 오히려 (-) … 모두 통계적으로 유의하지도 않았다'") for a, s in [("Jensen", "+"), ("3F", "+"), ("4F", "−")]]
주장 += [("시총5분위", "IMF", a, "+", "아님", "p.516 '(+)의 값을 나타냈지만 통계적으로 유의하진 않았다'") for a in ["Jensen", "3F", "4F"]]
판정 = []
for 구간, k, a, 방향, 유의, 글 in 주장:
    α, t = 우리표.loc[(구간, k), f"{a} α"], 우리표.loc[(구간, k), f"{a} t"]
    우리방향 = "+" if α > 0 else "−"
    우리유의 = next((n for n, x in 수준.items() if abs(t) >= x), "아님")
    맞음 = 우리방향 == 방향 and (abs(t) < 수준["10%"] if 유의 == "아님" else abs(t) >= 수준[유의])
    판정.append({"구간": 구간, "전략": k, "항목": f"{a} α", "원문 글": 글, "원문 방향": 방향, "원문 유의": 유의,
               "원문 α": 원문표.loc[(구간, k), f"{a} α"], "원문 t": 원문표.loc[(구간, k), f"{a} t"], "우리 α": α, "우리 t(OLS)": t,
               "우리 방향": 우리방향, "우리 유의(가장 강한 수준)": 우리유의, "원문과 같음": "같음" if 맞음 else "다름"})
for 구간 in ["전체"] + [f"시총{q}분위" for q in range(1, 6)]:  # p.514·515·517 '개선된 마법공식은 … 모두에서 기존의 마법공식보다 뛰어난 성과'
    a, b = 우리표.loc[(구간, "IMF"), "Return"], 우리표.loc[(구간, "MF"), "Return"]
    판정.append({"구간": 구간, "전략": "IMF vs MF", "항목": "Return(누적)", "원문 글": "p.517 '전체종목 … 시가총액 1분위에서 5분위까지 … 모두에서 기존의 마법공식보다 뛰어난 성과'",
               "원문 방향": "IMF > MF", "원문 α": 원문표.loc[(구간, "IMF"), "Return"], "원문 t": 원문표.loc[(구간, "MF"), "Return"], "우리 α": a, "우리 t(OLS)": b,
               "우리 방향": "IMF > MF" if a > b else "IMF ≤ MF", "원문과 같음": "같음" if a > b else "다름"})
# p.515–517 KOSPI 대비 문장(방향만). 비교한 숫자는 '원문 값'·'우리 값' 칸에 글로(KOSPI = 같은 기간 표 2 줄)
K = {"원문": 원문표.loc[("전체", "KOSPI")], "우리": 우리표.loc[("전체", "KOSPI")]}
값 = lambda 쪽, 구간, k, 칸들: ", ".join(f"{c} {(원문표 if 쪽 == '원문' else 우리표).loc[(구간, k), c]:.4f} vs KOSPI {K[쪽][c]:.4f}" for c in 칸들)
for k in ["MF", "IMF"]:  # 4분위: 누적·샤프로 보아 KOSPI보다 좋지 않음(원문 IMF는 누적은 KOSPI보다 크고 샤프 1.32 < 1.34 → '못하거나'를 '둘 다 앞서지는 않음'으로 읽음)
    좋음 = all(우리표.loc[("시총4분위", k), c] > K["우리"][c] for c in ["Return", "Sharpe"])
    판정.append({"구간": "시총4분위", "전략": f"{k} vs KOSPI", "항목": "Return·Sharpe",
               "원문 글": "p.515–516 '누적수익률과 샤프비율 등으로 확인하였을 때 4분위에서의 마법공식 및 개선된 마법공식은 벤치마크인 KOSPI 대비 좋은 성과를 나타내지 못하거나 오히려 나쁜 성과'",
               "원문 방향": "KOSPI보다 좋지 않음(누적·샤프 둘 다 앞서지는 않음)", "원문 값": 값("원문", "시총4분위", k, ["Return", "Sharpe"]), "우리 값": 값("우리", "시총4분위", k, ["Return", "Sharpe"]),
               "우리 방향": "누적·샤프 둘 다 KOSPI보다 큼" if 좋음 else "KOSPI보다 좋지 않음", "원문과 같음": "다름" if 좋음 else "같음"})
q4 = {n: 우리표.loc[("시총4분위", n), "Return"] for n in ["IMF", "MF"]} | {"KOSPI": K["우리"]["Return"]}
판정.append({"구간": "시총4분위", "전략": "IMF vs MF·KOSPI", "항목": "Return(누적)",
           "원문 글": "p.516 '하지만 개선된 마법공식은 여전히 기존 마법공식 및 벤치마크 보다 좋은 성과를 나타냈다'(4분위. 원문 표 6 IMF 샤프 1.32 < KOSPI 1.34라 누적으로 읽음)",
           "원문 방향": "IMF > MF, IMF > KOSPI", "원문 값": f"IMF {원문표.loc[('시총4분위', 'IMF'), 'Return']:.4f}, MF {원문표.loc[('시총4분위', 'MF'), 'Return']:.4f}, KOSPI {K['원문']['Return']:.4f}",
           "우리 값": ", ".join(f"{n} {v:.4f}" for n, v in q4.items()),
           "우리 방향": "IMF > MF, IMF > KOSPI" if q4["IMF"] > max(q4["MF"], q4["KOSPI"]) else "아님", "원문과 같음": "같음" if q4["IMF"] > max(q4["MF"], q4["KOSPI"]) else "다름"})
for 구간 in ["시총1분위", "시총2분위"]:  # p.516–517 1·2분위 'KOSPI 대비 누적수익률, 샤프비율, 정보비율 등 모든 면에서 월등하게 좋은 투자 성과'(정보비율은 KOSPI 대비라 KOSPI 줄 = 0)
    for k in ["MF", "IMF"]:
        좋음 = all(우리표.loc[(구간, k), c] > K["우리"][c] for c in ["Return", "Sharpe", "IR"])
        판정.append({"구간": 구간, "전략": f"{k} vs KOSPI", "항목": "Return·Sharpe·IR",
                   "원문 글": "p.516–517 '시가총액 하위 1분위 및 2분위, 즉 소형주들에서는 KOSPI 대비 누적수익률, 샤프비율, 정보비율 등 모든 면에서 월등하게 좋은 투자 성과'",
                   "원문 방향": "KOSPI보다 셋 다 큼", "원문 값": 값("원문", 구간, k, ["Return", "Sharpe", "IR"]), "우리 값": 값("우리", 구간, k, ["Return", "Sharpe", "IR"]),
                   "우리 방향": "KOSPI보다 셋 다 큼" if 좋음 else "셋 중 하나 이상 KOSPI 이하", "원문과 같음": "같음" if 좋음 else "다름"})
판정 = pd.DataFrame(판정)  # Return 줄은 '원문 α'·'우리 α' 칸 = IMF 누적, '원문 t'·'우리 t(OLS)' 칸 = MF 누적
판정.to_csv(평가.결과폴더 / "01_원문판정.csv", encoding="utf-8-sig", index=False)
print("원문 판정", len(판정), "줄 중 같음", int(판정["원문과 같음"].eq("같음").sum()))
판정

## 8. 표본: 원문 표 1(Number·Average return·Standard deviation) vs 우리
원문 3.1절의 표본 조건: 유가증권 + 코스닥, 은행·보험·증권·투자금융·신용금고·종합금융·기타금융 제외, 12월 결산, 필요한 자료 입수 가능, 기준일 거래정지 제외(p.512). '우리: 순위 매김' = 리밸런싱마다 거르기 뒤 MF 지표(ROC·EY)가 있어 순위를 매긴 종목 수.
우리 설정에는 원문에 없는 `보통주`(코드 끝자리 0, 회원 결정 "보통주만 해"), `쓰지말것`(README_데이터의 '쓰지 말 것' 구간) 단계도 있다. 투자기구(리츠·선박·부동산투자) 제외는 원문에 없어 뺐다(회원 05:57). 850에 맞추려고 조건을 새로 만들지 않는다. 단계별 남은 수는 확장연구.
평균수익률·표준편차: 원문은 내는 법을 적지 않았다. 우리는 순위를 매긴 종목마다 4월 첫 거래일 → 다음 해 3월 마지막 거래일 1년 수익률(수정주가, 중간 폐지는 마지막 상장 가격)의 단면 평균·표본 표준편차(%)로 잰다(기본값 ②, 11절). 그 1년 구간에 '쓰지 말 것' 칸이 걸린 종목은 뺀다(본문 수익률과 같은 규칙). 결과 `01_표1대조.csv`.

In [ ]:
표1 = pd.Series([850, 900, 972, 1033, 1098, 1129, 1196, 1282, 1351, 1382, 1434, 1484, 1568, 1638, 1744, 1815, 1880, 1955],
               index=range(2004, 2022), name="원문 표1")
기록 = 결과["MF"]["기록"]
표1통계 = pd.DataFrame({"원문 평균(%)": [40.88, 70.36, 11.60, 13.59, -26.56, 49.97, 14.02, 12.67, 14.41, 5.91, 32.31, 16.21, 3.40, 13.94, -8.43, -24.18, 98.69, 5.76],
                      "원문 표준편차(%)": [106.23, 187.52, 53.91, 67.75, 34.07, 87.05, 61.06, 61.35, 57.63, 43.52, 93.78, 61.97, 42.11, 62.35, 36.58, 42.81, 123.51, 49.53]}, index=표1.index)
P = 읽기.상장가격(가격)
Pf, d = P.ffill(), P.index.to_series()
마지막 = d.groupby(d.dt.to_period("M")).max()
금지 = 읽기.쓰지말것(가격)  # README_데이터 '쓰지 말 것' 구간(본문 수익률 계산과 같은 규칙)
우리1 = {}
for t in 기록.index:  # 리밸런싱일
    s, c, _ = 포.거르기(t, 설정["MF"])
    v = pd.DataFrame({m: getattr(지표, m)(s).reindex(c) for m, _ in 설정["MF"]["지표"]}).replace([np.inf, -np.inf], np.nan).dropna()
    끝날 = 마지막[pd.Period(f"{t.year + 1}-03", "M")]
    r = (Pf.loc[끝날] / P.loc[t] - 1).reindex(v.index)  # 4월 첫 거래일 → 다음 해 3월 마지막 거래일
    r[r.index.isin(금지.loc[(금지["시작"] < 끝날) & (금지["끝"] > t), "코드"])] = float("nan")  # 그 1년 구간에 '쓰지 말 것' 칸이 걸린 종목은 뺌
    우리1[t.year] = {"우리: 순위 매김": len(v), "우리 평균(%)": r.mean() * 100, "우리 표준편차(%)": r.std() * 100}
표1대조 = pd.concat([표1, 표1통계, pd.DataFrame(우리1).T], axis=1).dropna()
표1대조.to_csv(평가.결과폴더 / "01_표1대조.csv", encoding="utf-8-sig")
표1대조

### 남은 차이의 원인 후보
- **표 1 'Number'의 정의**: 원문은 '표본 종목 수', '분석대상 기업의 종목 수는 850개'(p.512)라고만 하고, 어떤 기준으로 셌는지(재료가 모두 있는 수인지, 상장 기간 조건이 있는지)는 적지 않았다.
- **원문 데이터(TS2000)의 종목 범위**: 코스닥 소형주·상장 직후 종목의 재무 입수 범위가 DataGuide와 다를 수 있다.
- **'필요한 자료 입수 가능'의 범위**: 우리는 t−1년 결산만 본다(원문 '직전년도'). 원문이 더 이전 연도까지 요구했는지는 알 수 없다.
- **금융업 범위**: 원문 7개 업종 이름이 거래소 업종(세부)과 1:1로 맞지 않는다. 거래소 업종이 빈 날은 이름 규칙(은행·보험·증권·금융·금고·캐피탈·카드·저축·신금)이나 그날 WICS '금융'으로 판정한다(업종 빈칸 수·WICS 금융 수는 확장연구 8절 기록 표).
- **이자비용 빈칸**: 이자비용이 없는 회사가 0 대신 빈칸으로 들어 있을 수 있다. 빈칸은 채우지 않고 그 종목을 뺀다(회원 결정, README '빈칸 그대로'. 연도별 빠진 수는 확장연구 6절)(빚 없는 회사가 빠지는 쪽으로 치우칠 수 있음).
- **원문에 적히지 않은 조건**: 우선주·관리종목·자본잠식 등이 TS2000 표본에서 빠져 있을 수 있다(확인 불가).
- **상장폐지 종목 포함 여부**: 원문에 서술이 없다.

## 9. 차이 이유 후보
- **데이터 출처**: 원문은 TS2000·DART·NAVER금융(p.512), 우리는 DataGuide. 과제 자료라 바꿀 수 없다(한계). 종목 범위·재무 값이 다를 수 있다(8절).
- **EBIT**: 세전계속사업이익 + 이자비용. 중단사업손익은 들어가지 않는다(원문이 중단사업을 어떻게 했는지 서술 없음).
- **표준편차·샤프·정보비율·MDD 정의**: 원문에 정의가 없다(p.513 식 (1)·(2)는 평균/표준편차 꼴만). KOSPI 원문 숫자(표준편차 83.49%, 샤프 1.34, MDD −54.54%, 정보비율 0.00)가 **일간 수익률, 표준편차 × √(전 기간 일 수), 샤프 = 일 초과수익(국고 3년 금리/365, 직전 날) 평균/표준편차 × √(일 수), MDD = 일간 가치**에서 나와(`_검증/01_정의역산.py`·`_결과.txt`) 이 정의로 잰다(7절 KOSPI 줄로 확인). p.514 '월별로 집계' 문장과는 맞지 않는다. 월간 × √12 판은 확장연구.
- **무위험 일간 /365**: 원문은 '3년 만기 국고채 수익률'(p.513)만 적었다. 거래일마다 연 금리/365를 빼면 1년에 빠지는 무위험이 연 금리보다 작다(거래일 약 250일). /252로 하면 KOSPI 샤프가 1.1444라 원문 1.34와 멀어져(`_검증/01_정의역산_결과.txt`) 원문 숫자가 나오는 /365를 쓴다.
- **누적수익률 계산**: 원문이 거래비용·월별 집계를 어떻게 했는지 자세히 나와 있지 않다. 우리는 리밸런싱 때 전량 매도·매수 비용을 그 뒤 첫 달에 뺀다(마지막 청산 비용은 안 뺌, 11절 기본값).
- **시장지수**: 코스피 가격지수(I.001). 원문은 가격/TR을 적지 않았지만, I.001 일간 종가 2004-04-01 → 2022-03-31 누적이 2.12393으로 원문 KOSPI 212.39%와 같다(`_검증/01_정의역산_결과.txt`). 포트폴리오도 수정주가(배당 미포함)라 정의가 맞는다. 코스닥 종목도 담지만 벤치마크는 코스피뿐이다.
- **요인**: 과제 공지('규모/가치/모멘텀 팩터 수익률은 직접 만들지 말고 DataGuide에서 제공하는 데이터를 사용')대로 DataGuide 요인(3FM.2B3.SMB·HML, 3FM.2M3.MOM, 2×3 요인지수)을 쓴다. 원문 p.513은 시총·BM·과거 11개월 수익률 상위 30% − 하위 30%로 직접 구성해 정의가 다르므로, 3요인·4요인 α 차이의 원인 후보다.
- **보유 중 상장폐지**: 원문 처리 방법이 없다. 우리는 폐지된 구간에 시장구분이 코스피·코스닥인 마지막 날 가격으로 판 것으로 보고, 그 뒤로는 남은 종목에 값 비례로 다시 넣은 것과 같게 계산한다(대가 지급 폐지는 반영 안 됨).

## 10. 원문과 다르게 한 것
| 항목 | 원문 | 우리 | 이유 |
|---|---|---|---|
| 데이터 | TS2000·DART·NAVER금융 | DataGuide(새로 통째로 받은 파일 `data\통합_전체_202611`, 상장일·폐지일·지수는 `data\DataGuide_전체_202611` X01·X03·X04) | 과제 데이터라 바꿀 수 없음(한계) |
| 상장 구간 | 상장 종목 | 시장구분(CP10000300)이 코스피·코스닥('KS'·'KQ')인 날의 가격만 계산에 씀(`읽기.상장가격`). 원본 값은 그대로 | 통합_확인기록 F2(상장일보다 앞선 가격 = 이전상장 전 시장 가격)·F3(상장폐지 뒤 장외 가격). 회원 결정 2026-11-02 |
| '쓰지 말 것' 칸 | — | 리밸런싱일이 README_데이터의 '쓰지 말 것' 구간 (시작, 끝]에 드는 종목은 편입 대상에서 뺌(`종목거르기.쓰지말것`, 항상 `시장` 다음 단계). 보유 중 걸린 칸은 TR 이상칸이면 그 구간 수정주가 수익률, 나머지는 빈칸(`포.쓰지말것칸`, 2026-11-04 고침: 전에는 다음 리밸런싱일까지 보고 뺌 = 미래 정보). 직전 12개월 일간수익률에서도 빈칸으로 봄. 값은 그대로 | A008080 2004-01~2013-09(D-05, 재조회 R28도 값 같음), 수정주가 병합·감자 미반영 7칸의 그 달(D27), 배당 포함 가격(TR)을 쓸 때 TR 이상칸의 앞날~그날(D-06·U3. 2026-11-04 같은 규칙으로 새 데이터에서 다시 만든 232칸, `_검증/새목록/`). 회원 결정 2026-11-02 |
| 호텔신라(A008770) FY2023 | — | 매출원가·매출총이익을 빈칸으로 봄 → 그해 GP/A 등에서 제외(값은 그대로, `지표.빈칸으로볼칸`) | 판본마다 값이 다름(R29 재조회는 매출원가 0, 10월 파일은 1,418,625,926천원). 통합_확인기록 F1. 회원 결정 2026-11-02 |
| 끝 날짜 | — | 가격·시장 일간 항목(S·CP 파일)에 값이 있는 마지막 날(`읽기.공통끝`)에서 자름. 재무 일간은 끝나지 않은 회계연도가 빈칸이라 판정에서 뺌 | 통합_확인기록 F6(항목마다 끝 날짜가 다를 수 있음) |
| 코드 재사용 16종목 | — | 재사용 경계를 넘는 수익률을 쓰지 않음(공백 구간이 빈칸이라 끊김, `연구/따라하기에서_옮김_01.ipynb` 7절에서 확인). 직전 12개월 상장 판정도 시장구분으로 함 | 통합_확인기록 F5, README_데이터 D-04 |
| 재무 값 뽑기 | — | 달력기준 일간 재무에서 결산일(결산월 이력이 그 달인 달의 마지막 영업일) 값을 그 회계연도 값으로 씀(`읽기.연간재무`). 쓸 수 있는 날은 재무시점 규칙 | 일간 값은 발표 전부터 그 회계연도 값이 들어 있어 그대로 쓰면 미래 정보. 예전 연간 원본과 자산총계 79,657/79,837행 같음(`_검증/일간재무_대조_결과.txt`) |
| 같은 회계연도 두 결산월 | — | 결산일이 늦은 행 | 통합_확인기록 F9(A432320) |
| A070480 신비앤텍 | — | 값 그대로. 상위 30에 한 번도 안 들어 뺀 판은 본문과 같아 두지 않음(`연구/따라하기에서_옮김_01.ipynb` 7절에서 보유 횟수 0 확인) | 한 달 큰 수익률(같은 노트북 7절). 회원 결정 2026-11-02, 2026-11-04 중복 정리 |
| 법인세차감전이익 | 법인세차감전이익 | 세전계속사업이익(M000907001). **중단사업손익은 들어가지 않음** | K-IFRS 법인세비용차감전순이익은 계속사업 기준이고 중단사업손익은 세후로 따로 표시. 중단사업이익 항목(M000907005 등)은 받은 재무 파일에 없어, 중단사업이 있는 종목-연도 수는 셀 수 없음 |
| 금융업 판정 | 은행업·보험업·증권업·투자금융업·신용금고업·종합금융업·기타금융업(p.512) | 그날 거래소 업종명(CP10001090)에 은행·보험·증권·금융·금고가 있으면 뺌. 거래소 업종이 빈 날은 이름(지금 이름)에 은행·보험·증권·금융·금고·캐피탈·카드·저축·신금이 있거나 그날 WICS 대분류(CP10000600)가 '금융'이면 뺌(`종목거르기.금융업(이름="원문")`) | 원문 7업종 이름이 거래소 업종(세부)과 1:1로 맞지 않고 업종이 빈 날이 있어, 빈 날은 보조 자료(이름·WICS, WICS는 2000년부터)로 대응(기본값 11절). 리츠·스팩·'지주'로 끝남은 7업종이 아니라 이름 규칙에서 뺌(회원 05:57) |
| 투자기구 | 서술 없음 | 거르지 않음 | 원문에 없어 뺌(회원 05:57). 연도별 투자기구 이름에 걸리는 종목 수는 `연구/따라하기에서_옮김_01.ipynb` 6절 |
| 이자비용 빈칸 | 필요한 자료 입수 가능한 기업만 | 빈칸 그대로, 그 종목은 뺌 | 원본 값만 씀. 연도별 빠진 수는 `연구/따라하기에서_옮김_01.ipynb` 6절 |
| 결산월 | 12월 결산법인 | 회계연도별 결산월 이력(CP10002200) | 결산월이 바뀐 회사를 그해 결산월로 판정 |
| 재무 시점 | 3월 말까지 공시된 직전 연도 결산 | 결산일 + 3개월, 회계연도 t−1 결산만(없으면 그해 빠짐) | 원문 그대로. 실제 발표일 판(확인판)은 확장연구 |
| GP/A | 매출총이익 / 총자산 | 같음: 매출총이익(M000904007) / 자산총계 | 원문 p.513 |
| 요인 | 시총·BM·과거 11개월 수익률 상위 30% − 하위 30% 직접 구성(p.513) | DataGuide 요인(3FM.2B3.SMB·HML, 3FM.2M3.MOM, `평가.요인`) | 과제 공지대로 DataGuide 요인 사용. 그래서 원문 3·4요인 α와 정의가 다름 |
| SD·Sharpe·IR·MDD | 정의 없음 | 일간 정의(`평가.일간정의`, 무위험 국고 3년 일간/365) | KOSPI 원문 숫자가 이 정의에서 나옴(9절) |
| 보유 중 폐지 | 언급 없음 | 폐지 구간은 시장구분이 코스피·코스닥인 마지막 날 가격으로 정산, 그 뒤엔 남은 종목에 비례 재투자 | 폐지 손실을 빠뜨리지 않기. 폐지 뒤 장외 가격은 안 씀(F3) |
| 거래비용 차감 | 전량 매도·매수 | 같음(`전량교체=True`). 리밸런싱 뒤 첫 달에 뺌, 마지막 청산 비용은 안 뺌 | 가장 단순한 방식 |


## 11. 원문에 말이 없어 정한 것 (기본값)
원문에 없는데 빼면 계산이 안 되는 칸은 회원 결정(2026-11-06)대로 ① 자료가 없거나 계산이 안 되면 그 종목을 그 달(해)에서 빼고 ② 나머지는 가장 단순한 쪽으로 정했다. 다른 선택의 영향은 이 노트북에서 재지 않는다. 원문에 근거가 있는 줄은 굵게 따로 적었다(**원문에 있음**·**원문 안 어긋남**·**역산(원문 숫자에 맞춰 찾은 정의)**·**제도로 정함**·**해석**).

| 항목 | 원문(쪽) | 쓴 기본 방법 | 칸 |
|---|---|---|---|
| 3월 말 → 4월 첫 거래일 하루 | **원문 안 어긋남**: p.512 '4월 첫 거래일에 기존 포트폴리오를 전량 매도하고, 새로운 재무데이터를 활용하여 투자 포트폴리오를 매입' 대 같은 쪽 '분석대상의 기간은 매년 4월 첫 거래일부터 다음연도 3월 마지막 거래일까지로 구분' | 포트폴리오는 앞 문장을 따름(같은 날 팔고 사므로 3월 말~4월 첫 거래일 하루도 보유). 표 1 평균수익률·표준편차(8절)는 뒤 문장을 따름(4월 첫 거래일 → 다음 해 3월 마지막 거래일, 기간 정의 일치) | 4절 `포.실행`, 8절 |
| 월수익률 재는 날 | p.514 '월별로 집계'(어느 날로 나누는지 없음) | 월 첫 거래일 간격(리밸런싱일과 같은 날, ②) | 4절 `포.기준일` |
| 분위를 나누는 모집단 | p.510·514 '시가총액 기준 5분위', p.512 표본 = '입수 가능한 기업' | 그 전략 지표가 있는 종목(①) | 6절 `지표있음` |
| 분위 시점·나누는 법·분위 안 순위 | 시점은 **원문에 있음(간접 근거)**: p.512 '4월 첫 거래일 시점 기준의 종가에 맞춘 데이터'(마법공식 절차 '첫째' 단계 문장이라 분위 시점을 따로 말한 것은 아님). 나누는 법·분위 안 순위는 없음 | 4월 첫 거래일 시총(원문), 종목 수 같게·분위 안에서 순위 다시 매김(②) | 6절 `시총분위` |
| MF·IMF 표본 | p.512 표 1 표본 수 하나, p.513 '기타 사항은 마법공식과 동일' | 전략마다 그 지표가 있는 종목(①) | 4절 `포.실행` |
| 같은 순위 처리 | 없음 | 순위 합이 같으면 표의 앞 종목(`nlargest` 기본, ②) | `포.고르기` |
| EV 시가총액 | p.512 '시가총액'(우선주 여부 없음) | 그 코드(보통주) 시가총액만(②) | `지표.기업가치` |
| 현금 | p.512 '현금' | 현금및현금성자산(M000901030, ②) | `지표.현금` |
| 무형자산 빈칸 | 없음 | 그 종목을 뺌(①) | `지표.ROC` |
| 분모가 0 이하 | 없음 | 공식대로 계산(±무한대만 빈칸, ②) | `포.실행` |
| 거래세를 매도에만 | p.512 '증권거래세 0.25%, 매수매도 거래 시마다 추가 거래비용(Slippage) 0.5%'(거래세를 내는 시점 말 없음) | **제도로 정함**: 증권거래세는 매도에 부과되므로 매도만, 수수료·슬리피지는 양쪽 | `포.거래비용` |
| 비용 빼는 때·마지막 정리 비용 | 없음 | 리밸런싱 뒤 첫 달에 뺌, 2022-03-31 마지막 정리 비용은 안 뺌(②) | `포.보유` |
| 보유 중 상장폐지 | 없음 | 마지막 상장 가격에 팔고 남은 종목에 비례 재투자(②) | `포.보유` |
| 무위험 월 환산 | p.513 '3년 만기 국고채 수익률' | 연 %/1200, 직전 달 말 값(②) | 4절 `평가.요인` |
| 무위험 일간 환산 | 위와 같음 | **역산(원문 숫자에 맞춰 찾은 정의)**: 연 %/100/365, 직전 날 값(/365에서 KOSPI 원문 샤프 1.34가 나옴, /252면 1.1444, 9절) | 4절 `무위험일` |
| SD·Sharpe·IR·MDD 정의 | p.513 정의 없음(식 (1)·(2)는 평균/표준편차 꼴만) | **역산(원문 숫자에 맞춰 찾은 정의)**: 일간 수익률, SD = 일 표준편차 × √N, 샤프 = 일 초과수익 평균/표준편차 × √N, IR = (일 수익률 − KOSPI) 평균/표준편차 × √N, MDD = 일간 가치(원문 KOSPI 0.8349·1.34·−54.54%가 나옴, 9절) | 4절 `평가.일간정의` |
| 요인 | p.513 SMB·HML·MOM을 30%/30%로 직접 구성 | 과제 공지대로 DataGuide 요인(3FM.2B3.SMB·HML, 3FM.2M3.MOM) 사용. 원문 3·4요인 α와 정의가 다름(직접 구성용 기본값 = 가중·주기·장부가·재무 시점·30% 종목 수 등은 쓰지 않음) | 4절 `평가.요인` |
| 일간 표준편차 자유도 | 없음 | 표본 표준편차(ddof = 1, pandas 기본, ②) | `평가.일간정의` |
| 국고 3년 일간의 빈 날 | 없음 | X02에 값이 없는 날(V01 검토로 254일)은 앞 날 값으로 채움(ffill, ②) | 4절 `무위험일` |
| 재무 '결산일 + 3개월' | **원문에 있음**: p.512 '연간 사업보고서가 공시되는 3월 말까지 데이터를 수집하고 4월 첫 거래일에 … 리밸런싱'(발표가 늦은 회사 말 없음) | 발표일을 보지 않고 3월 말이 지나면 FY t−1을 씀. 실제 발표가 4월 첫 거래일 뒤인 종목도 그날 씀(원문 규칙을 따라도 생기는 미래 정보, 한계) | `포.재무` |
| 거래소 업종이 빈 날의 금융업 판정 | p.512 금융 7업종(업종 자료가 빈 날 말 없음) | 원문 7업종을 보조 자료로 대응: 이름에 은행·보험·증권·금융·금고·캐피탈·카드·저축·신금이 있거나 그날 WICS '금융'이면 뺌 | `종목거르기.금융업(이름="원문")` |
| 원문 판정 KOSPI 대비 7줄의 읽는 법 | p.515–516 4분위 '누적수익률과 샤프비율 등으로 확인하였을 때 … KOSPI 대비 좋은 성과를 나타내지 못하거나 오히려 나쁜 성과', p.516 '개선된 마법공식은 여전히 기존 마법공식 및 벤치마크 보다 좋은 성과', p.516–517 1·2분위 'KOSPI 대비 누적수익률, 샤프비율, 정보비율 등 모든 면에서 월등하게 좋은 투자 성과'(어느 칸으로 견주는지 글이 모자람) | **해석**: ① 4분위 '못하거나' = 누적·샤프 둘 다 KOSPI를 앞서지는 않음 ② 4분위 IMF '여전히 … 좋은 성과' = 누적만(원문 표 6 IMF 샤프 1.32 < KOSPI 1.34) ③ 1·2분위 '모든 면에서' = 이름이 나온 셋(누적·샤프·IR)만, MDD는 뺌(원문 표 3 1분위 MF·IMF MDD −0.5760·−0.6203이 KOSPI −0.5454보다 나쁨) | 7절 `01_원문판정.csv`(코드 칸 14) |
| 표 1 평균수익률·표준편차 내는 법 | p.512 표 1(내는 법 없음) | 순위 매긴 종목의 4월 첫 거래일 → 다음 3월 마지막 거래일 1년 수익률 단면 평균·표본 표준편차(②) | 8절 |

## 12. 요약표용 저장
4절 상위 30 포트폴리오(본문 EW, MF·IMF)의 월수익률과 요인(DataGuide)을 `결과_월수익률/01_우동호2023.pkl`에 남긴다. 요약표 노트북(`연구/요약표.ipynb`)이 읽는다. 시험 모드면 `_검증/1106/시험결과/`에.

In [ ]:
R2 = R.drop(columns="KOSPI")
R2.columns = pd.MultiIndex.from_tuples([(c, "동일") for c in R2])
평가.저장(R2, F, "01_우동호2023")
list(R2)